# DARA — All-in-One Google Colab Integration (Cloud-only, free-first)
**GPT coordinates. Google Drive is the canonical Knowledge & Data Spine. GitHub is the Technology & Code Spine. Google Colab/Google Cloud execute.**

**Project:** `third-serenity-428705-g2`  
**Existing DARA Google Drive folder:** https://drive.google.com/drive/folders/1OXZ6v1zB0G7wmlGezLCAvm8I4Egr2UPL  
**Existing DARA GitHub:** https://github.com/Saehon/Saeid-Homayoun/tree/main/DARA

### What this notebook does
1. Mount your existing Google Drive (with your explicit Google consent).
2. Verify direct Drive readback without replacing any research files.
3. Authenticate Google Cloud; inspect enabled services (no changes to APIs).
4. Check GitHub code availability (read-only).
5. Test **read-only** Google Drive REST metadata access when OAuth scopes permit; optionally probe Docs/Sheets metadata of *already existing* DARA files.
6. Test **read-only** BigQuery dataset metadata and Vertex AI custom-model metadata.
7. Detect whether *optional* Gemini and OpenAI API keys are stored in Colab Secrets, without printing or using them.
8. Save a versioned status report to the **same canonical Drive folder**, and verify readback.

### Important boundaries
- **No Google service APIs are enabled**, no cloud resources are provisioned, and no SQL or AI generation requests are run.
- **Gemini in the Colab sidebar cannot automatically gain Workspace access from this notebook.** Use the copy-ready master prompt below; access remains bound to the sidebar's actual tools and permissions.
- **Gemini CLI / Google Workspace CLI in Cloud Shell require a separate terminal and OAuth flow**, not controlled by this Colab session.
- **Gmail and Calendar:** this notebook inspects API states only; it never opens messages or events.
- **No installation on your Windows PC.** Python packages install only into the temporary Colab runtime.
- A `PASS` for metadata is not evidence of operational model inference or permission to query BigQuery data.
- Prior DARA research project folders are not moved or duplicated.

Select **Runtime → Run all**. There are no blocking `input()` calls; Google consent prompts are still necessary.


## 1 — Install only the Python dependencies used by Colab


In [ ]:
import sys, subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "google-auth", "google-cloud-bigquery", "google-genai", "requests"],
    check=True,
    timeout=180
)
print("Colab runtime dependencies installed. No packages installed on Windows.")


## 2 — Initialize DARA state, non-secret run ID and configuration


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen
from google.auth.transport.requests import AuthorizedSession
import google.auth, json, os, platform, uuid, subprocess

PROJECT_ID = "third-serenity-428705-g2"
GITHUB_REPO = "Saehon/Saeid-Homayoun"
DARA_DRIVE_FOLDER_ID = "1OXZ6v1zB0G7wmlGezLCAvm8I4Egr2UPL"
DRIVE_FOLDER = Path("/content/drive/MyDrive/DARA_ResearchOS")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
statuses = {}
enabled_services = None
credentials = None
drive_file_metadata = []

def record(test, status, detail=""):
    statuses[test] = {"status": status, "detail": str(detail)[:250]}
    print(f"{test}: {status}" + (f" | {detail}" if detail else ""))

print("DARA run:", RUN_ID, "Python:", platform.python_version())
print("Cloud project:", PROJECT_ID)


## 3 — Google Drive direct mount and non-destructive readback


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert DRIVE_FOLDER.parent.is_dir(), "Drive mount unavailable: MyDrive was not found"
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)

marker_file = DRIVE_FOLDER / f"DARA_ALL_IN_ONE_READBACK_{RUN_ID}.txt"
marker_text = f"DARA direct Drive verification: {RUN_ID}"
with marker_file.open("x", encoding="utf-8") as f:
    f.write(marker_text)
assert marker_file.read_text(encoding="utf-8") == marker_text
record("Google Drive mounted and readback", "PASS", marker_file.name)

# Show names only (no research file contents).
visible = [p.name for p in list(DRIVE_FOLDER.iterdir())[:30]]
record("DARA canonical folder", "PASS", "Mounted, existing directory accessible")
print("DARA folder items (up to 30):", visible)


## 4 — Google Cloud authentication and exact project check


In [ ]:
from google.colab import auth
auth.authenticate_user()
proc = subprocess.run(
    ["gcloud", "config", "set", "project", PROJECT_ID],
    capture_output=True, text=True, timeout=30
)
if proc.returncode != 0:
    record("Google Cloud ADC / project", "FAIL", proc.stderr[-180:])
    raise RuntimeError("Google Cloud project selection failed; cannot run cloud tests.")
actual_project = subprocess.check_output(
    ["gcloud", "config", "get-value", "project"], text=True, timeout=25
).strip()
credentials, adc_project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
assert credentials is not None and actual_project == PROJECT_ID
session = AuthorizedSession(credentials)
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
record("Google Cloud ADC / project", "PASS", "Credentials available, gcloud project matches")
print("Note: this alone does NOT verify IAM to each service or Gemini model inference.")


## 5 — GitHub code spine, read-only


In [ ]:
url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/DARA/ARCHITECTURE.md"
try:
    with urlopen(Request(url, headers={"User-Agent":"DARA-Colab-Audit"}), timeout=15) as res:
        item = json.load(res)
    assert item["path"] == "DARA/ARCHITECTURE.md"
    record("GitHub code bridge", "PASS", "DARA architecture readable (no writes)")
except Exception as exc:
    record("GitHub code bridge", "NOT VERIFIED", type(exc).__name__)


## 6 — Inspect which Google Cloud APIs are enabled (no enablement)


In [ ]:
SERVICES = {
    "Drive API": "drive.googleapis.com",
    "Docs API": "docs.googleapis.com",
    "Sheets API": "sheets.googleapis.com",
    "Gmail API": "gmail.googleapis.com",
    "Calendar API": "calendar-json.googleapis.com",
    "BigQuery API": "bigquery.googleapis.com",
    "Vertex AI API": "aiplatform.googleapis.com",
    "Cloud Storage API": "storage.googleapis.com",
}
try:
    proc = subprocess.run(
        ["gcloud", "services", "list", "--enabled", "--project", PROJECT_ID,
         "--format=json"],
        capture_output=True, text=True, timeout=35, check=True
    )
    enabled_services = set()
    for item in json.loads(proc.stdout):
        name = item.get("config", {}).get("name") or item.get("name", "")
        enabled_services.add(name.split("/")[-1])
    for label, service in SERVICES.items():
        record(label, "ENABLED" if service in enabled_services else "DISABLED",
               "API state only; not proof of user authorization")
except Exception as exc:
    record("Cloud API inventory", "NOT VERIFIED", type(exc).__name__)
    for label in SERVICES:
        record(label, "UNKNOWN", "Service Usage list unavailable")


## 7 — Google Drive REST API: test access to the existing DARA folder

This tests access **through Google Cloud ADC**, which is not the same as the Drive mount. If it returns 401/403 or insufficient scopes, your direct mounted Drive access can still be working. Do **not** interpret it as a failed Drive mount. It may require a separate Google Workspace OAuth consent/client.


In [ ]:
try:
    response = session.get(
        "https://www.googleapis.com/drive/v3/files/" + DARA_DRIVE_FOLDER_ID,
        params={"fields": "id,name,mimeType", "supportsAllDrives": "true"},
        timeout=20,
    )
    if response.ok:
        info = response.json()
        assert info.get("id") == DARA_DRIVE_FOLDER_ID
        record("Drive REST metadata", "PASS", "Folder metadata fetched through ADC")
        listed = session.get(
            "https://www.googleapis.com/drive/v3/files",
            params={
                "q": f"'{DARA_DRIVE_FOLDER_ID}' in parents and trashed = false",
                "fields": "files(id,name,mimeType),nextPageToken",
                "pageSize": 50
            },
            timeout=20
        )
        if listed.ok:
            drive_file_metadata = listed.json().get("files", [])
            record("Drive REST files.list", "PASS",
                   str(len(drive_file_metadata)) + " file metadata rows")
        else:
            record("Drive REST files.list", "BLOCKED",
                   f"HTTP {listed.status_code}; Workspace OAuth scopes/API may be needed")
    else:
        record("Drive REST metadata", "BLOCKED",
               f"HTTP {response.status_code}; API disabled, insufficient scopes or IAM")
except Exception as exc:
    record("Drive REST metadata", "NOT VERIFIED", type(exc).__name__)


## 8 — Google Docs and Sheets: read existing document metadata only

Checks the first native Docs/Sheets file in the DARA folder **if Drive REST returned its file metadata**. No file contents are edited or copied. Missing files mean NOT TESTED rather than failure.


In [ ]:
targets = [
    ("Docs document metadata", "application/vnd.google-apps.document",
     lambda fileid: "https://docs.googleapis.com/v1/documents/" + fileid,
     {"fields":"documentId,title"}),
    ("Sheets spreadsheet metadata", "application/vnd.google-apps.spreadsheet",
     lambda fileid: "https://sheets.googleapis.com/v4/spreadsheets/" + fileid,
     {"fields":"spreadsheetId,properties(title)"}),
]
for name, mime, urlfunc, params in targets:
    found = next((x for x in drive_file_metadata if x.get("mimeType") == mime), None)
    if not found:
        record(name, "NOT TESTED", "No corresponding file discovered via Drive REST")
        continue
    try:
        res = session.get(urlfunc(found["id"]), params=params, timeout=20)
        record(name, "PASS" if res.ok else "BLOCKED",
               "HTTP " + str(res.status_code) + "; metadata only")
    except Exception as exc:
        record(name, "NOT VERIFIED", type(exc).__name__)


## 9 — BigQuery: read-only datasets metadata (no SQL query)


In [ ]:
if enabled_services is not None and "bigquery.googleapis.com" not in enabled_services:
    record("BigQuery metadata", "SKIPPED", "API appears disabled")
else:
    try:
        from google.cloud import bigquery
        bq = bigquery.Client(project=PROJECT_ID, credentials=credentials)
        datasets = list(bq.list_datasets(max_results=1, timeout=20))
        record("BigQuery metadata", "PASS",
               "datasets.list worked; no SQL job; sample count=" + str(len(datasets)))
    except Exception as exc:
        record("BigQuery metadata", "BLOCKED/NOT VERIFIED",
               type(exc).__name__ + ": " + str(exc)[:100])


## 10 — Vertex AI: read-only custom-models metadata and optional Gemini SDK configuration

`models.list` on the custom-model metadata endpoint is **not** a Gemini text-generation request. No paid generation is performed; an SDK client can be configured without requesting inference.


In [ ]:
if enabled_services is not None and "aiplatform.googleapis.com" not in enabled_services:
    record("Vertex AI custom-model metadata", "SKIPPED", "API appears disabled")
    record("Gemini Vertex SDK", "NOT VERIFIED", "Vertex API is disabled or not configured")
else:
    try:
        location = "us-central1"
        endpoint = (
            f"https://{location}-aiplatform.googleapis.com/v1/"
            f"projects/{PROJECT_ID}/locations/{location}/models?pageSize=1"
        )
        res = session.get(endpoint, timeout=20)
        record("Vertex AI custom-model metadata", "PASS" if res.ok else "BLOCKED",
               f"models.list HTTP {res.status_code}; Gemini inference NOT tested")
        from google import genai
        client = genai.Client(
            vertexai=True, project=PROJECT_ID, location="global",
            credentials=credentials
        )
        record("Gemini Vertex SDK", "CONFIGURED",
               "Client initialized; inference NOT requested")
        client.close()
    except Exception as exc:
        record("Vertex AI custom-model metadata", "NOT VERIFIED", type(exc).__name__)


## 11 — Optional OpenAI and Gemini Developer API configuration (no inference)

ChatGPT's Drive/GitHub app connections do **not** use your Colab keys.

If you separately need Colab to call GPT or Gemini Developer API, store keys in **Colab → Secrets** (`OPENAI_API_KEY` or `GEMINI_API_KEY`). This cell only checks whether secrets exist. It never prints, saves or uses either key. All paid model generation remains disabled.


In [ ]:
from google.colab import userdata
for label, key in [("OpenAI API secret", "OPENAI_API_KEY"),
                   ("Gemini Developer API secret", "GEMINI_API_KEY")]:
    exists = False
    try:
        value = userdata.get(key)
        exists = bool(value)
        value = None
    except Exception:
        pass
    record(label, "PRESENT — NOT USED" if exists else "NOT CONFIGURED",
           "Optional for model calls from Colab, not ChatGPT connectors")
record("Gemini in Colab sidebar", "MANUAL ONLY",
       "This notebook cannot inspect sidebar conversation/authorizations")
record("Gemini CLI in Cloud Shell", "SEPARATE SESSION",
       "This notebook cannot authenticate or control Cloud Shell")


## 12 — Save final DARA verification report to Google Drive and read it back

This is a **new** JSON file; previous artifacts and research data are untouched. No credentials or private file contents are saved. Every API status is explicitly labelled with its test scope.


In [ ]:
report = {
    "system": "DARA — Dual-Spine AI Research Architecture",
    "stage": "Colab All-in-One Cloud and Workspace readiness",
    "run_id": RUN_ID,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "google_cloud_project": PROJECT_ID,
    "github_repository": GITHUB_REPO,
    "drive_folder_id": DARA_DRIVE_FOLDER_ID,
    "status": statuses,
    "limitations": [
        "Colab Drive mount does not grant Gemini sidebar or Gemini CLI universal Drive access.",
        "Google Workspace CLI is separately authorized in Cloud Shell.",
        "No Cloud APIs, billing/IAM, dataset content, SQL jobs or model inference were changed/executed.",
        "BigQuery and Vertex AI metadata success does not prove operational queries or Gemini generation.",
        "Gmail and Calendar content was not accessed.",
        "No API key material was saved."
    ]
}
path = DRIVE_FOLDER / f"DARA_ALL_IN_ONE_VERIFICATION_{RUN_ID}.json"
with path.open("x", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
saved = json.loads(path.read_text(encoding="utf-8"))
assert saved["run_id"] == RUN_ID and saved["google_cloud_project"] == PROJECT_ID
print("DARA REPORT SAVED AND READ BACK:", path)
print("\n=== VERIFICATION RESULTS ===")
for key, value in statuses.items():
    print(f"{key:32} {value['status']}")
print("\nSend the report name to ChatGPT to verify via its Google Drive connector.")


# 13 — Master prompt for Gemini in Colab (copy manually into Gemini sidebar)

**Important:** Gemini's sidebar must have the proper connected tools and identity to inspect Drive documents. Pasting a prompt alone does not grant permissions. Do not claim actual access unless Gemini returns cited evidence.

> You are **DARA ResearchOS Integration Architect**, assisting in a Google Colab notebook. DARA stands for **Dual-Spine AI Research Architecture**. GPT/ChatGPT is the Research Intelligence & Coordination layer; Google Drive is the canonical **Knowledge & Data Spine** with vertical project folders and horizontal shared research methods; GitHub is the **Technology & Code Spine**. Colab, Gemini, Cloud Shell and Google Cloud are separate execution environments with separate permissions.
>
> Use only the Google Drive, Docs, Sheets, BigQuery or Vertex AI tools **already authorized** in this environment. The existing Drive folder is **DARA_ResearchOS**, ID `1OXZ6v1zB0G7wmlGezLCAvm8I4Egr2UPL`. Google Cloud project: `third-serenity-428705-g2`. GitHub: `Saehon/Saeid-Homayoun`.
>
> **Task:** (1) Examine the visible outputs of this notebook and the latest `DARA_ALL_IN_ONE_VERIFICATION_*.json` file only if genuinely accessible. (2) Identify which services passed verified tests and which are still unauthorized/untested. (3) Explain precisely how to grant **least-privilege** read-only access to Drive, Docs, and Sheets without enabling any unrelated APIs. (4) Preserve all existing research folders and their content. (5) Distinguish Colab's Drive mount, Google Workspace CLI OAuth, ChatGPT connectors and Gemini sidebar capabilities. (6) Do not ask me to paste access tokens or secrets, do not activate billing/services, do not commit to GitHub, and do not infer that a metadata check proves paid model availability.
>
> **Deliverable:** A concise PASS/PENDING/BLOCKED integration matrix, grounded in actual notebook results, with only the minimum next steps. Do not invent completed operations.


# 14 — Separate terminal: Gemini CLI + Workspace CLI (Google Cloud Shell, NOT Colab)

Open [Google Cloud Console](https://console.cloud.google.com/), click **Activate Cloud Shell**, then run the *read-only* diagnostics:

```bash
gcloud config set project third-serenity-428705-g2
gcloud config get-value project
node --version
npm --version
gemini --version
command -v gws || true
gcloud services list --enabled --project=third-serenity-428705-g2 --format="value(config.name)"
```

**Only after your approval and confirming the original package:** `npm install -g @googleworkspace/cli` (installs only in Cloud Shell).

**Google Workspace OAuth needs separate approval and an existing project's OAuth client;** do **not** run `gws auth setup` unreviewed because it may create projects and enable APIs. Use `gws auth login -s drive` after configuration; perform read-only file listings first. The CLI can be extended to Gemini CLI using `gemini extensions install https://github.com/googleworkspace/cli`, after reviewing the extension source and permissions.

Gemini CLI login and Gemini Colab UI are **not** verified by running this notebook. Gmail, Calendar and other Workspace authorizations remain separate. If a service is disabled, do not enable it automatically.

After executing the notebook, return to ChatGPT and request: **"Review the latest DARA_ALL_IN_ONE_VERIFICATION JSON in my DARA_ResearchOS Google Drive folder and explain remaining integration gates."**
